# Temporal Feature Engineering for TIM v2

Analysis-only notebook for qualifying temporal interaction features on IEMOCAP and MELD. It does not train the main SER models and does not use transcript text as model input.


## 1. Setup


In [5]:
from __future__ import annotations

import json
import math
import os
import sys
import warnings
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
os.environ.setdefault("MPLCONFIGDIR", str(Path("/tmp") / "matplotlib"))
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import mutual_info_classif
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import LabelEncoder, StandardScaler

try:
    import scipy.stats as st
    SCIPY_AVAILABLE = True
except Exception as exc:
    st = None
    SCIPY_AVAILABLE = False
    warnings.warn(f"scipy unavailable; statistical tests will be partially skipped: {exc}")

try:
    import seaborn as sns
    SEABORN_AVAILABLE = True
except Exception:
    sns = None
    SEABORN_AVAILABLE = False

try:
    import torch
    TORCH_AVAILABLE = True
except Exception as exc:
    torch = None
    TORCH_AVAILABLE = False
    warnings.warn(f"torch unavailable; WavLM complementarity probing may be skipped: {exc}")

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
OUT_DIR = PROJECT_ROOT / "results" / "feature_engineering"
PLOT_DIR = OUT_DIR / "plots"
DIST_PLOT_DIR = PLOT_DIR / "distributions"
ASSOC_PLOT_DIR = PLOT_DIR / "emotion_association"
REDUNDANCY_PLOT_DIR = PLOT_DIR / "redundancy"
for path in [OUT_DIR, DIST_PLOT_DIR, ASSOC_PLOT_DIR, REDUNDANCY_PLOT_DIR]:
    path.mkdir(parents=True, exist_ok=True)

LABEL_ORDER = ["angry", "happy", "neutral", "sad"]
LABEL_TO_ID = {label: idx for idx, label in enumerate(LABEL_ORDER)}
MELD_LABEL_MAP = {
    "anger": "angry", "angry": "angry", "disgust": "angry", "disgusted": "angry",
    "joy": "happy", "happy": "happy", "surprise": "happy",
    "sadness": "sad", "sad": "sad", "fear": "sad", "neutral": "neutral",
}
print(f"PROJECT_ROOT={PROJECT_ROOT}")
print(f"OUT_DIR={OUT_DIR}")


PROJECT_ROOT=/Users/ngocbao/Documents/Document/research/main/speech/exps/demo
OUT_DIR=/Users/ngocbao/Documents/Document/research/main/speech/exps/demo/results/feature_engineering


## 2. Load Metadata


In [6]:
def standardize_iemocap_from_utils(root: Path) -> pd.DataFrame:
    try:
        from utils.iemocap_kaggle import discover_iemocap_samples
    except Exception as exc:
        warnings.warn(f"Cannot import discover_iemocap_samples: {exc}")
        return pd.DataFrame()
    try:
        samples = discover_iemocap_samples(root, auto_download=False)
    except Exception as exc:
        warnings.warn(f"Cannot load IEMOCAP from {root}: {exc}")
        return pd.DataFrame()
    rows = []
    for sample in samples:
        rows.append({
            "dataset": "IEMOCAP",
            "dialogue_id": sample.dialogue_id,
            "utterance_id": sample.utterance_id,
            "speaker_id": sample.speaker_id,
            "turn_index": np.nan,
            "start_time": float(sample.start_time),
            "end_time": float(sample.end_time),
            "duration": max(0.0, float(sample.end_time) - float(sample.start_time)),
            "label": sample.label_name,
            "audio_path": sample.audio_path,
            "transcript_text": sample.transcript,
        })
    return pd.DataFrame(rows)


def parse_meld_time(value):
    if pd.isna(value):
        return np.nan
    text = str(value).strip().replace(",", ".")
    if not text:
        return np.nan
    parts = text.split(":")
    try:
        if len(parts) == 3:
            h, m, s = parts
            return float(h) * 3600 + float(m) * 60 + float(s)
        if len(parts) == 2:
            m, s = parts
            return float(m) * 60 + float(s)
        return float(text)
    except Exception:
        return np.nan


def find_meld_csvs(root_candidates):
    paths = []
    for root in root_candidates:
        root = Path(root)
        if root.exists():
            paths.extend(sorted(root.rglob("*sent_emo.csv")))
            paths.extend(sorted(root.rglob("*sent_emo*.csv")))
    return sorted(set(paths))


def infer_meld_split(path: Path) -> str:
    text = str(path).lower()
    if "train" in text:
        return "train"
    if "dev" in text or "valid" in text:
        return "validation"
    if "test" in text:
        return "test"
    return path.stem


def standardize_meld_from_csvs(root_candidates) -> pd.DataFrame:
    csvs = find_meld_csvs(root_candidates)
    rows = []
    for csv_path in csvs:
        try:
            frame = pd.read_csv(csv_path)
        except Exception as exc:
            warnings.warn(f"Cannot read MELD CSV {csv_path}: {exc}")
            continue
        cols = {c.lower(): c for c in frame.columns}
        dialogue_col = cols.get("dialogue_id") or cols.get("dialogueid")
        utterance_col = cols.get("utterance_id") or cols.get("utteranceid")
        emotion_col = cols.get("emotion")
        speaker_col = cols.get("speaker")
        text_col = cols.get("utterance")
        start_col = cols.get("starttime") or cols.get("start_time")
        end_col = cols.get("endtime") or cols.get("end_time")
        if dialogue_col is None or utterance_col is None or emotion_col is None:
            continue
        split = infer_meld_split(csv_path)
        for _, row in frame.iterrows():
            raw_label = str(row[emotion_col]).strip().lower()
            label = MELD_LABEL_MAP.get(raw_label)
            if label is None:
                continue
            dialogue_num = row[dialogue_col]
            utterance_num = row[utterance_col]
            start_time = parse_meld_time(row[start_col]) if start_col else np.nan
            end_time = parse_meld_time(row[end_col]) if end_col else np.nan
            rows.append({
                "dataset": "MELD",
                "dialogue_id": f"meld_{split}_dia{dialogue_num}",
                "utterance_id": f"meld_{split}_dia{dialogue_num}_utt{utterance_num}",
                "speaker_id": str(row[speaker_col]) if speaker_col else "unknown",
                "turn_index": int(utterance_num) if pd.notna(utterance_num) else np.nan,
                "start_time": start_time,
                "end_time": end_time,
                "duration": max(0.0, end_time - start_time) if pd.notna(start_time) and pd.notna(end_time) else np.nan,
                "label": label,
                "audio_path": "",
                "transcript_text": str(row[text_col]) if text_col else "",
            })
    return pd.DataFrame(rows)


def load_prediction_metadata() -> pd.DataFrame:
    rows = []
    for path in sorted((PROJECT_ROOT / "results").glob("**/predictions.csv")):
        try:
            frame = pd.read_csv(path)
        except Exception:
            continue
        rename = {}
        if "true_label" in frame.columns and "label" not in frame.columns:
            rename["true_label"] = "label"
        if "label_name" in frame.columns and "label" not in frame.columns:
            rename["label_name"] = "label"
        frame = frame.rename(columns=rename)
        required_any = {"dialogue_id", "utterance_id"}
        if not required_any.issubset(frame.columns):
            continue
        keep = [c for c in ["dialogue_id", "utterance_id", "speaker_id", "start_time", "end_time", "label", "audio_path", "transcript_text"] if c in frame.columns]
        tmp = frame[keep].copy()
        tmp["dataset"] = "prediction_cache"
        rows.append(tmp)
    if not rows:
        return pd.DataFrame()
    pred = pd.concat(rows, ignore_index=True).drop_duplicates(["dialogue_id", "utterance_id"])
    return pred


iemocap_roots = [PROJECT_ROOT / "data" / "iemocap", PROJECT_ROOT / "iemocap"]
meld_roots = [PROJECT_ROOT / "data" / "meld", PROJECT_ROOT / "data" / "meld-dataset", PROJECT_ROOT / "meld", PROJECT_ROOT / "MELD"]
frames = []
for root in iemocap_roots:
    if root.exists():
        df_i = standardize_iemocap_from_utils(root)
        if not df_i.empty:
            frames.append(df_i)
            print(f"Loaded IEMOCAP: {len(df_i)} rows from {root}")
            break

df_m = standardize_meld_from_csvs(meld_roots)
if not df_m.empty:
    frames.append(df_m)
    print(f"Loaded MELD metadata: {len(df_m)} rows from local CSV files")

if not frames:
    pred_meta = load_prediction_metadata()
    if not pred_meta.empty:
        frames.append(pred_meta)
        print(f"Loaded fallback prediction metadata: {len(pred_meta)} rows")

if frames:
    df_all = pd.concat(frames, ignore_index=True, sort=False)
else:
    df_all = pd.DataFrame()

required_metadata_cols = [
    "dataset", "dialogue_id", "utterance_id", "speaker_id",
    "turn_index", "start_time", "end_time", "duration",
    "label", "audio_path", "transcript_text",
]
for col in required_metadata_cols:
    if col not in df_all.columns:
        df_all[col] = np.nan if col in {"turn_index", "start_time", "end_time", "duration"} else ""

if df_all.empty:
    warnings.warn(
        "No metadata loaded. Check data/iemocap, data/meld, or prediction cache paths."
    )

for col in ["dataset", "dialogue_id", "utterance_id", "speaker_id", "label", "audio_path", "transcript_text"]:
    df_all[col] = df_all[col].fillna("").astype(str)
for col in ["start_time", "end_time", "duration", "turn_index"]:
    df_all[col] = pd.to_numeric(df_all[col], errors="coerce")

def assign_turn_indices(group):
    if group["start_time"].notna().any():
        group = group.sort_values(["start_time", "end_time", "utterance_id"], kind="mergesort")
    else:
        group = group.sort_values(["turn_index", "utterance_id"], kind="mergesort")
    group = group.copy()
    group["turn_index"] = np.arange(len(group))
    return group

ordered_groups = []
for _, group in df_all.groupby(["dataset", "dialogue_id"], sort=False, dropna=False):
    ordered_groups.append(assign_turn_indices(group))

if ordered_groups:
    df_all = pd.concat(ordered_groups, ignore_index=True)
else:
    df_all = df_all.iloc[0:0].copy()

df_all["duration"] = np.where(df_all["duration"].notna(), df_all["duration"], df_all["end_time"] - df_all["start_time"])
df_all.to_csv(OUT_DIR / "metadata_all.csv", index=False)
print(df_all.groupby("dataset").size())
df_all.head()


Loaded IEMOCAP: 7529 rows from /Users/ngocbao/Documents/Document/research/main/speech/exps/demo/data/iemocap
Loaded MELD metadata: 13708 rows from local CSV files
dataset
IEMOCAP     7529
MELD       13708
dtype: int64


,dataset,dialogue_id,utterance_id,speaker_id,turn_index,start_time,end_time,duration,label,audio_path,transcript_text
0,IEMOCAP,Ses01F_impro01,Ses01F_impro01_F000,Ses01_F,0,6.2901,8.2357,1.9456,neutral,/Users/ngocbao/Documents/Document/research/mai...,Excuse me.
1,IEMOCAP,Ses01F_impro01,Ses01F_impro01_M000,Ses01_M,1,7.5712,10.4750,2.9038,angry,/Users/ngocbao/Documents/Document/research/mai...,Do you have your forms?
2,IEMOCAP,Ses01F_impro01,Ses01F_impro01_F001,Ses01_F,2,10.0100,11.3925,1.3825,neutral,/Users/ngocbao/Documents/Document/research/mai...,Yeah.
3,IEMOCAP,Ses01F_impro01,Ses01F_impro01_M001,Ses01_M,3,10.9266,14.6649,3.7383,angry,/Users/ngocbao/Documents/Document/research/mai...,Let me see them.
4,IEMOCAP,Ses01F_impro01,Ses01F_impro01_F002,Ses01_F,4,14.8872,18.0175,3.1303,neutral,/Users/ngocbao/Documents/Document/research/mai...,Is there a problem?


## 3. Dataset Feasibility Table


In [7]:
FEATURE_REQUIREMENTS = {
    "response_latency": ("Response Dynamics", ["start_time", "end_time"]),
    "relative_response_latency": ("Response Dynamics", ["start_time", "end_time", "speaker_id"]),
    "latency_trend": ("Response Dynamics", ["start_time", "end_time"]),
    "latency_variance": ("Response Dynamics", ["start_time", "end_time"]),
    "immediate_response": ("Response Dynamics", ["start_time", "end_time"]),
    "short_response": ("Response Dynamics", ["start_time", "end_time"]),
    "long_pause": ("Response Dynamics", ["start_time", "end_time"]),
    "speaker_switch": ("Turn-taking", ["speaker_id"]),
    "consecutive_turn_count": ("Turn-taking", ["speaker_id"]),
    "turn_holding_duration": ("Turn-taking", ["speaker_id", "start_time", "end_time"]),
    "turn_yielding": ("Turn-taking", ["speaker_id", "start_time", "end_time"]),
    "speaker_switch_frequency_window": ("Turn-taking", ["speaker_id"]),
    "overlap_duration": ("Overlap / Interruption", ["start_time", "end_time"]),
    "overlap_ratio": ("Overlap / Interruption", ["start_time", "end_time"]),
    "overlap_flag": ("Overlap / Interruption", ["start_time", "end_time"]),
    "strong_overlap": ("Overlap / Interruption", ["start_time", "end_time"]),
    "interruption_flag": ("Overlap / Interruption", ["speaker_id", "start_time", "end_time"]),
    "consecutive_overlap_count": ("Overlap / Interruption", ["start_time", "end_time"]),
    "overlap_frequency_window": ("Overlap / Interruption", ["start_time", "end_time"]),
    "competitive_overlap_proxy": ("Overlap / Interruption", ["speaker_id", "start_time", "end_time"]),
    "cooperative_overlap_proxy": ("Overlap / Interruption", ["speaker_id", "start_time", "end_time"]),
    "interaction_density": ("Dialogue Rhythm", ["start_time", "end_time"]),
    "silence_density": ("Dialogue Rhythm", ["start_time", "end_time"]),
    "burstiness": ("Dialogue Rhythm", ["start_time", "end_time"]),
    "rhythm_variance": ("Dialogue Rhythm", ["start_time", "end_time"]),
    "speaker_dominance_time": ("Speaker Behavior", ["speaker_id", "start_time", "end_time"]),
    "speaker_dominance_turns": ("Speaker Behavior", ["speaker_id"]),
    "speaker_interruption_tendency": ("Speaker Behavior", ["speaker_id", "start_time", "end_time"]),
    "speaker_yield_tendency": ("Speaker Behavior", ["speaker_id", "start_time", "end_time"]),
    "speaker_response_habit": ("Speaker Behavior", ["speaker_id", "start_time", "end_time"]),
    "speaker_persistence": ("Speaker Behavior", ["speaker_id"]),
    "rapid_exchange_state": ("Dialogue State", ["speaker_id", "start_time", "end_time"]),
    "conflict_like_state": ("Dialogue State", ["speaker_id", "start_time", "end_time"]),
    "hesitation_state": ("Dialogue State", ["start_time", "end_time"]),
    "calm_state": ("Dialogue State", ["start_time", "end_time"]),
    "floor_competition_state": ("Dialogue State", ["speaker_id", "start_time", "end_time"]),
}

def dataset_support(dataset_name: str, required: list[str]) -> tuple[str, str]:
    subset = df_all[df_all["dataset"] == dataset_name]
    if subset.empty:
        return "no", "dataset unavailable"
    missing = [c for c in required if c not in subset.columns or subset[c].isna().all()]
    if missing:
        return "no", f"missing/unavailable: {missing}"
    if any(subset[c].isna().mean() > 0.3 for c in required if c in subset.columns):
        return "partial", "required columns exist but have high missing rate"
    if dataset_name == "MELD" and {"start_time", "end_time"}.intersection(required):
        return "partial", "timestamps available in CSV but may be clip-local and less reliable than IEMOCAP"
    return "yes", "available"

feasibility_rows = []
for feature_name, (group, required) in FEATURE_REQUIREMENTS.items():
    i_status, i_reason = dataset_support("IEMOCAP", required)
    m_status, m_reason = dataset_support("MELD", required)
    uses_future = False
    feasibility_rows.append({
        "feature_name": feature_name,
        "phenomenon_group": group,
        "required_columns": ", ".join(required),
        "computable_on_IEMOCAP": i_status,
        "iemocap_reason": i_reason,
        "computable_on_MELD": m_status,
        "meld_reason": m_reason,
        "causal": True,
        "uses_future_information": uses_future,
        "suitable_for_online_inference": True,
    })
feasibility = pd.DataFrame(feasibility_rows)
feasibility.to_csv(OUT_DIR / "feature_feasibility.csv", index=False)
feasibility.head(20)


,feature_name,phenomenon_group,required_columns,computable_on_IEMOCAP,iemocap_reason,computable_on_MELD,meld_reason,causal,uses_future_information,suitable_for_online_inference
0,response_latency,Response Dynamics,"start_time, end_time",yes,available,partial,timestamps available in CSV but may be clip-lo...,True,False,True
1,relative_response_latency,Response Dynamics,"start_time, end_time, speaker_id",yes,available,partial,timestamps available in CSV but may be clip-lo...,True,False,True
2,latency_trend,Response Dynamics,"start_time, end_time",yes,available,partial,timestamps available in CSV but may be clip-lo...,True,False,True
3,latency_variance,Response Dynamics,"start_time, end_time",yes,available,partial,timestamps available in CSV but may be clip-lo...,True,False,True
4,immediate_response,Response Dynamics,"start_time, end_time",yes,available,partial,timestamps available in CSV but may be clip-lo...,True,False,True
5,short_response,Response Dynamics,"start_time, end_time",yes,available,partial,timestamps available in CSV but may be clip-lo...,True,False,True
6,long_pause,Response Dynamics,"start_time, end_time",yes,available,partial,timestamps available in CSV but may be clip-lo...,True,False,True
7,speaker_switch,Turn-taking,speaker_id,yes,available,yes,available,True,False,True
8,consecutive_turn_count,Turn-taking,speaker_id,yes,available,yes,available,True,False,True
9,turn_holding_duration,Turn-taking,"speaker_id, start_time, end_time",yes,available,partial,timestamps available in CSV but may be clip-lo...,True,False,True


## 4. Temporal Feature Construction


In [8]:
TEMPORAL_CONFIG = {
    "overlap_threshold": 0.05,
    "strong_overlap_ratio_threshold": 0.30,
    "immediate_gap_threshold": 0.10,
    "short_gap_threshold": 0.30,
    "long_gap_threshold": 1.00,
    "rapid_exchange_window": 3,
    "rhythm_window": 5,
    "density_window_seconds": 10.0,
    "eps": 1e-8,
}

BINARY_FEATURES = {
    "immediate_response", "short_response", "long_pause", "speaker_switch", "same_speaker_continuation",
    "overlap_flag", "strong_overlap", "interruption_flag", "competitive_overlap_proxy", "cooperative_overlap_proxy",
    "rapid_exchange_state", "conflict_like_state", "hesitation_state", "calm_state", "floor_competition_state",
}

FEATURE_GROUPS = {
    "duration": "Basic",
    "gap_prev": "Basic",
    "overlap_duration": "Overlap / Interruption",
    "overlap_ratio": "Overlap / Interruption",
    "abs_gap": "Basic",
    "log_turn_index": "Basic",
    "immediate_response": "Response Dynamics",
    "short_response": "Response Dynamics",
    "long_pause": "Response Dynamics",
    "relative_gap_to_speaker_mean": "Response Dynamics",
    "previous_mean_gap": "Response Dynamics",
    "window3_average_gap": "Response Dynamics",
    "window5_average_gap": "Response Dynamics",
    "window3_gap_variance": "Response Dynamics",
    "window5_gap_variance": "Response Dynamics",
    "speaker_switch": "Turn-taking",
    "same_speaker_continuation": "Turn-taking",
    "consecutive_same_speaker_turns": "Turn-taking",
    "turn_holding_duration_so_far": "Turn-taking",
    "speaker_switch_frequency_window3": "Turn-taking",
    "speaker_switch_frequency_window5": "Turn-taking",
    "overlap_flag": "Overlap / Interruption",
    "strong_overlap": "Overlap / Interruption",
    "interruption_flag": "Overlap / Interruption",
    "consecutive_overlap_count": "Overlap / Interruption",
    "overlap_frequency_window3": "Overlap / Interruption",
    "overlap_frequency_window5": "Overlap / Interruption",
    "interruption_frequency_window3": "Overlap / Interruption",
    "interruption_frequency_window5": "Overlap / Interruption",
    "competitive_overlap_proxy": "Overlap / Interruption",
    "cooperative_overlap_proxy": "Overlap / Interruption",
    "interaction_density_10s": "Dialogue Rhythm",
    "silence_density_10s": "Dialogue Rhythm",
    "burstiness_window5": "Dialogue Rhythm",
    "rhythm_variance_window5": "Dialogue Rhythm",
    "speaker_prev_turn_count": "Speaker Behavior",
    "speaker_prev_total_speaking_time": "Speaker Behavior",
    "speaker_dominance_time_so_far": "Speaker Behavior",
    "speaker_dominance_turns_so_far": "Speaker Behavior",
    "speaker_prev_overlap_rate": "Speaker Behavior",
    "speaker_prev_interruption_rate": "Speaker Behavior",
    "speaker_prev_mean_gap": "Speaker Behavior",
    "speaker_prev_mean_duration": "Speaker Behavior",
    "speaker_persistence_so_far": "Speaker Behavior",
    "rapid_exchange_state": "Dialogue State",
    "conflict_like_state": "Dialogue State",
    "hesitation_state": "Dialogue State",
    "calm_state": "Dialogue State",
    "floor_competition_state": "Dialogue State",
}


def rolling_mean(values, window):
    values = list(values)[-window:]
    return float(np.nanmean(values)) if values else np.nan


def rolling_var(values, window):
    values = list(values)[-window:]
    return float(np.nanvar(values)) if len(values) > 1 else 0.0


def build_temporal_features(df_dataset: pd.DataFrame, config: dict = TEMPORAL_CONFIG) -> pd.DataFrame:
    frames = []
    ts_cols = ["start_time", "end_time"]
    timestamp_supported = all(c in df_dataset.columns for c in ts_cols) and df_dataset[ts_cols].notna().all(axis=1).mean() > 0.7
    for dialogue_id, group in df_dataset.groupby("dialogue_id", sort=False):
        group = group.sort_values(["turn_index", "utterance_id"], kind="mergesort").copy()
        speaker_stats = defaultdict(lambda: {"turns": 0, "time": 0.0, "overlaps": 0, "interruptions": 0, "gap_sum": 0.0, "duration_sum": 0.0, "same_continuations": 0})
        total_time_so_far = 0.0
        total_turns_so_far = 0
        previous_rows = []
        gaps, overlaps, interruptions, switches = [], [], [], []
        consecutive_overlap = 0
        consecutive_same_speaker = 0
        records = []
        for local_idx, (_, row) in enumerate(group.iterrows()):
            speaker = str(row.get("speaker_id", "unknown"))
            duration = float(row["duration"]) if timestamp_supported and pd.notna(row.get("duration")) else np.nan
            if not np.isfinite(duration):
                duration = max(0.0, float(row["end_time"] - row["start_time"])) if timestamp_supported else np.nan
            has_prev = len(previous_rows) > 0
            prev = previous_rows[-1] if has_prev else None
            if timestamp_supported and has_prev:
                gap_prev = float(row["start_time"] - prev["end_time"])
                overlap_duration = max(0.0, float(prev["end_time"] - row["start_time"]))
            elif timestamp_supported:
                gap_prev = 0.0
                overlap_duration = 0.0
            else:
                gap_prev = np.nan
                overlap_duration = np.nan
            overlap_ratio = overlap_duration / max(float(duration) if np.isfinite(duration) else 0.0, config["eps"]) if timestamp_supported else np.nan
            speaker_switch = 1.0 if has_prev and speaker != str(prev["speaker_id"]) else 0.0
            same_speaker = 1.0 if has_prev and speaker == str(prev["speaker_id"]) else 0.0
            consecutive_same_speaker = consecutive_same_speaker + 1 if same_speaker else 0
            overlap_flag = 1.0 if timestamp_supported and overlap_duration > config["overlap_threshold"] else 0.0
            strong_overlap = 1.0 if timestamp_supported and overlap_ratio >= config["strong_overlap_ratio_threshold"] else 0.0
            interruption_flag = 1.0 if speaker_switch and overlap_flag else 0.0
            consecutive_overlap = consecutive_overlap + 1 if overlap_flag else 0
            gaps.append(gap_prev)
            overlaps.append(overlap_flag)
            interruptions.append(interruption_flag)
            switches.append(speaker_switch)
            hist = speaker_stats[speaker]
            previous_mean_gap = float(np.nanmean(gaps[:-1])) if len(gaps) > 1 else np.nan
            speaker_prev_mean_gap = hist["gap_sum"] / max(hist["turns"], 1)
            speaker_prev_mean_duration = hist["duration_sum"] / max(hist["turns"], 1)
            relative_gap = gap_prev - speaker_prev_mean_gap if timestamp_supported and hist["turns"] > 0 else np.nan
            recent = previous_rows[-20:]
            if timestamp_supported:
                now = float(row["start_time"])
                window_rows = [r for r in recent if now - float(r["start_time"]) <= config["density_window_seconds"]]
                interaction_density = (len(window_rows) + 1) / config["density_window_seconds"]
                recent_gaps = [max(0.0, g) for g in gaps[-5:] if pd.notna(g)]
                silence_density = sum(recent_gaps) / config["density_window_seconds"]
            else:
                interaction_density = np.nan
                silence_density = np.nan
            switch_freq_3 = rolling_mean(switches, 3)
            switch_freq_5 = rolling_mean(switches, 5)
            overlap_freq_3 = rolling_mean(overlaps, 3)
            overlap_freq_5 = rolling_mean(overlaps, 5)
            int_freq_3 = rolling_mean(interruptions, 3)
            int_freq_5 = rolling_mean(interruptions, 5)
            w5_avg_gap = rolling_mean(gaps, 5)
            w5_gap_var = rolling_var(gaps, 5)
            burstiness = (float(np.nanstd(gaps[-5:])) / (abs(float(np.nanmean(gaps[-5:]))) + config["eps"])) if timestamp_supported and len(gaps[-5:]) > 1 else np.nan
            rapid_exchange = 1.0 if timestamp_supported and pd.notna(w5_avg_gap) and w5_avg_gap < config["short_gap_threshold"] and switch_freq_5 >= 0.5 else 0.0
            conflict_like = 1.0 if overlap_freq_5 >= 0.4 or int_freq_5 >= 0.3 else 0.0
            hesitation = 1.0 if timestamp_supported and (gap_prev > config["long_gap_threshold"] or (pd.notna(w5_avg_gap) and w5_avg_gap > config["long_gap_threshold"])) else 0.0
            calm = 1.0 if timestamp_supported and overlap_freq_5 < 0.1 and config["short_gap_threshold"] <= max(gap_prev, 0.0) <= config["long_gap_threshold"] and (not pd.notna(w5_gap_var) or w5_gap_var < 0.5) else 0.0
            floor_comp = 1.0 if consecutive_overlap >= 2 or (speaker_switch and timestamp_supported and gap_prev < 0.0 and switch_freq_3 >= 0.66) else 0.0
            record = {
                **{c: row.get(c, np.nan) for c in ["dataset", "dialogue_id", "utterance_id", "speaker_id", "turn_index", "label", "audio_path", "transcript_text", "start_time", "end_time"]},
                "duration": duration,
                "gap_prev": gap_prev,
                "overlap_duration": overlap_duration,
                "overlap_ratio": overlap_ratio,
                "abs_gap": abs(gap_prev) if timestamp_supported else np.nan,
                "log_turn_index": np.log1p(local_idx),
                "immediate_response": 1.0 if timestamp_supported and has_prev and 0 <= gap_prev < config["immediate_gap_threshold"] else 0.0,
                "short_response": 1.0 if timestamp_supported and has_prev and 0 <= gap_prev < config["short_gap_threshold"] else 0.0,
                "long_pause": 1.0 if timestamp_supported and has_prev and gap_prev > config["long_gap_threshold"] else 0.0,
                "relative_gap_to_speaker_mean": relative_gap,
                "previous_mean_gap": previous_mean_gap,
                "window3_average_gap": rolling_mean(gaps, 3),
                "window5_average_gap": w5_avg_gap,
                "window3_gap_variance": rolling_var(gaps, 3),
                "window5_gap_variance": w5_gap_var,
                "speaker_switch": speaker_switch,
                "same_speaker_continuation": same_speaker,
                "consecutive_same_speaker_turns": consecutive_same_speaker,
                "turn_holding_duration_so_far": hist["duration_sum"],
                "speaker_switch_frequency_window3": switch_freq_3,
                "speaker_switch_frequency_window5": switch_freq_5,
                "overlap_flag": overlap_flag,
                "strong_overlap": strong_overlap,
                "interruption_flag": interruption_flag,
                "consecutive_overlap_count": consecutive_overlap,
                "overlap_frequency_window3": overlap_freq_3,
                "overlap_frequency_window5": overlap_freq_5,
                "interruption_frequency_window3": int_freq_3,
                "interruption_frequency_window5": int_freq_5,
                "competitive_overlap_proxy": 1.0 if speaker_switch and overlap_duration > config["overlap_threshold"] and overlap_ratio >= config["strong_overlap_ratio_threshold"] else 0.0,
                "cooperative_overlap_proxy": 1.0 if speaker_switch and overlap_duration > config["overlap_threshold"] and overlap_ratio < config["strong_overlap_ratio_threshold"] else 0.0,
                "interaction_density_10s": interaction_density,
                "silence_density_10s": silence_density,
                "burstiness_window5": burstiness,
                "rhythm_variance_window5": w5_gap_var,
                "speaker_prev_turn_count": hist["turns"],
                "speaker_prev_total_speaking_time": hist["time"],
                "speaker_dominance_time_so_far": hist["time"] / max(total_time_so_far, config["eps"]),
                "speaker_dominance_turns_so_far": hist["turns"] / max(total_turns_so_far, 1),
                "speaker_prev_overlap_rate": hist["overlaps"] / max(hist["turns"], 1),
                "speaker_prev_interruption_rate": hist["interruptions"] / max(hist["turns"], 1),
                "speaker_prev_mean_gap": speaker_prev_mean_gap,
                "speaker_prev_mean_duration": speaker_prev_mean_duration,
                "speaker_persistence_so_far": hist["same_continuations"] / max(hist["turns"], 1),
                "rapid_exchange_state": rapid_exchange,
                "conflict_like_state": conflict_like,
                "hesitation_state": hesitation,
                "calm_state": calm,
                "floor_competition_state": floor_comp,
            }
            records.append(record)
            hist["turns"] += 1
            hist["time"] += 0.0 if not np.isfinite(duration) else duration
            hist["overlaps"] += overlap_flag
            hist["interruptions"] += interruption_flag
            hist["gap_sum"] += 0.0 if not timestamp_supported or not np.isfinite(gap_prev) else gap_prev
            hist["duration_sum"] += 0.0 if not np.isfinite(duration) else duration
            hist["same_continuations"] += same_speaker
            speaker_stats[speaker] = hist
            total_turns_so_far += 1
            total_time_so_far += 0.0 if not np.isfinite(duration) else duration
            previous_rows.append(record)
        frames.append(pd.DataFrame(records))
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

features_by_dataset = []
for dataset_name, frame in df_all.groupby("dataset", sort=False):
    features_by_dataset.append(build_temporal_features(frame, TEMPORAL_CONFIG))
features_all = pd.concat(features_by_dataset, ignore_index=True) if features_by_dataset else pd.DataFrame()
feature_cols = [c for c in FEATURE_GROUPS if c in features_all.columns]
features_all.to_csv(OUT_DIR / "features_all.csv", index=False)
print(features_all.groupby("dataset").size() if not features_all.empty else "No features built")
features_all[["dataset", "utterance_id", "label"] + feature_cols[:10]].head()


dataset
IEMOCAP     7529
MELD       13708
dtype: int64


,dataset,utterance_id,label,duration,gap_prev,overlap_duration,overlap_ratio,abs_gap,log_turn_index,immediate_response,short_response,long_pause,relative_gap_to_speaker_mean
0,IEMOCAP,Ses01F_impro01_F000,neutral,1.9456,0.0000,0.0000,0.000000,0.0000,0.000000,0.0,0.0,0.0,NaN
1,IEMOCAP,Ses01F_impro01_M000,angry,2.9038,-0.6645,0.6645,0.228838,0.6645,0.693147,0.0,0.0,0.0,NaN
2,IEMOCAP,Ses01F_impro01_F001,neutral,1.3825,-0.4650,0.4650,0.336347,0.4650,1.098612,0.0,0.0,0.0,-0.4650
3,IEMOCAP,Ses01F_impro01_M001,angry,3.7383,-0.4659,0.4659,0.124629,0.4659,1.386294,0.0,0.0,0.0,0.1986
4,IEMOCAP,Ses01F_impro01_F002,neutral,3.1303,0.2223,0.0000,0.000000,0.2223,1.609438,0.0,1.0,0.0,0.4548


## 5. Distribution Analysis


In [9]:
def is_binary_series(series: pd.Series) -> bool:
    vals = pd.Series(series.dropna().unique())
    return len(vals) <= 2 and set(vals.astype(float).round(8)).issubset({0.0, 1.0}) if len(vals) else False

rows = []
for dataset_name, frame in features_all.groupby("dataset", sort=False):
    for feature in feature_cols:
        s = pd.to_numeric(frame[feature], errors="coerce")
        valid = s.dropna()
        if valid.empty:
            stats = {"count": 0, "missing_rate": 1.0, "mean": np.nan, "std": np.nan, "median": np.nan, "min": np.nan, "max": np.nan, "q05": np.nan, "q25": np.nan, "q75": np.nan, "q95": np.nan, "unique_count": 0, "zero_rate": np.nan, "binary_rate": np.nan}
        else:
            is_binary = is_binary_series(valid)
            stats = {
                "count": int(valid.shape[0]),
                "missing_rate": float(s.isna().mean()),
                "mean": float(valid.mean()),
                "std": float(valid.std(ddof=0)),
                "median": float(valid.median()),
                "min": float(valid.min()),
                "max": float(valid.max()),
                "q05": float(valid.quantile(0.05)),
                "q25": float(valid.quantile(0.25)),
                "q75": float(valid.quantile(0.75)),
                "q95": float(valid.quantile(0.95)),
                "unique_count": int(valid.nunique()),
                "zero_rate": float((valid == 0).mean()),
                "binary_rate": float(valid.mean()) if is_binary else np.nan,
            }
        imbalance = max(stats.get("zero_rate", 0) or 0, 1 - (stats.get("zero_rate", 0) or 0)) if stats.get("unique_count", 0) <= 2 else np.nan
        stats.update({
            "dataset": dataset_name,
            "feature": feature,
            "phenomenon_group": FEATURE_GROUPS.get(feature, "Unknown"),
            "is_binary": bool(is_binary_series(valid)) if not valid.empty else False,
            "near_constant": bool((stats.get("zero_rate", 0) or 0) > 0.95 or (stats.get("unique_count", 99) <= 2 and (imbalance if pd.notna(imbalance) else 0) > 0.95)),
            "sparse_event": bool(pd.notna(stats.get("binary_rate", np.nan)) and stats["binary_rate"] < 0.05),
            "high_missing": bool(stats.get("missing_rate", 0) > 0.30),
        })
        rows.append(stats)

distribution_stats = pd.DataFrame(rows)
distribution_stats.to_csv(OUT_DIR / "feature_distribution_stats.csv", index=False)

for dataset_name, frame in features_all.groupby("dataset", sort=False):
    for feature in feature_cols:
        s = pd.to_numeric(frame[feature], errors="coerce").dropna()
        if s.empty:
            continue
        plt.figure(figsize=(5, 3))
        if is_binary_series(s):
            s.value_counts().sort_index().plot(kind="bar")
            plt.ylabel("count")
        else:
            s.clip(s.quantile(0.01), s.quantile(0.99)).plot(kind="hist", bins=30)
            plt.ylabel("count")
        plt.title(f"{dataset_name}: {feature}")
        plt.tight_layout()
        plt.savefig(DIST_PLOT_DIR / f"{dataset_name}_{feature}.png", dpi=120)
        plt.close()

distribution_stats.head()


,count,missing_rate,mean,std,median,min,max,q05,q25,q75,...,unique_count,zero_rate,binary_rate,dataset,feature,phenomenon_group,is_binary,near_constant,sparse_event,high_missing
0,7529,0.0,4.557893,3.157419,3.61000,0.585,34.138800,1.410,2.3416,5.850000,...,5358,0.000000,NaN,IEMOCAP,duration,Basic,False,False,False,False
1,7529,0.0,0.706608,4.243834,-0.22000,-18.190,107.590000,-2.326,-0.6900,0.640000,...,4784,0.030416,NaN,IEMOCAP,gap_prev,Basic,False,False,False,False
2,7529,0.0,0.580158,1.160873,0.22000,0.000,18.190000,0.000,0.0000,0.690000,...,2339,0.453447,NaN,IEMOCAP,overlap_duration,Overlap / Interruption,False,False,False,False
3,7529,0.0,0.207723,0.531773,0.04642,0.000,11.444444,0.000,0.0000,0.217742,...,4105,0.453447,NaN,IEMOCAP,overlap_ratio,Overlap / Interruption,False,False,False,False
4,7529,0.0,1.866924,3.876082,0.68000,0.000,107.590000,0.020,0.3175,1.712600,...,4317,0.030416,NaN,IEMOCAP,abs_gap,Basic,False,False,False,False


## 6. Emotion Association Analysis


In [10]:
def cramers_v_from_table(table: pd.DataFrame) -> float:
    if not SCIPY_AVAILABLE:
        return np.nan
    chi2 = st.chi2_contingency(table)[0]
    n = table.to_numpy().sum()
    if n <= 0:
        return np.nan
    r, k = table.shape
    return float(np.sqrt((chi2 / n) / max(min(k - 1, r - 1), 1)))

assoc_rows, classwise_rows = [], []
for dataset_name, frame in features_all.dropna(subset=["label"]).groupby("dataset", sort=False):
    y = frame["label"].astype(str)
    label_ids = LabelEncoder().fit_transform(y)
    for feature in feature_cols:
        x = pd.to_numeric(frame[feature], errors="coerce")
        valid_mask = x.notna() & y.notna()
        if valid_mask.sum() < 10 or y[valid_mask].nunique() < 2:
            continue
        xv = x[valid_mask]
        yv = y[valid_mask]
        yid = LabelEncoder().fit_transform(yv)
        is_binary = is_binary_series(xv)
        p_value = np.nan
        effect_size = np.nan
        test_name = ""
        if is_binary:
            table = pd.crosstab(yv, xv.astype(int))
            if SCIPY_AVAILABLE and table.shape[1] >= 2:
                p_value = float(st.chi2_contingency(table)[1])
                effect_size = cramers_v_from_table(table)
                test_name = "chi_square"
            for label, sub in pd.DataFrame({"label": yv, "x": xv}).groupby("label"):
                classwise_rows.append({"dataset": dataset_name, "feature": feature, "label": label, "mean_or_rate": float(sub["x"].mean()), "count": int(len(sub))})
        else:
            groups = [g.dropna().values for _, g in pd.DataFrame({"label": yv, "x": xv}).groupby("label")["x"]]
            if SCIPY_AVAILABLE and len(groups) >= 2 and all(len(g) > 1 for g in groups):
                kw = st.kruskal(*groups)
                p_value = float(kw.pvalue)
                n = sum(len(g) for g in groups)
                k = len(groups)
                effect_size = float(max(0.0, (kw.statistic - k + 1) / max(n - k, 1)))
                test_name = "kruskal_wallis_epsilon2"
            for label, sub in pd.DataFrame({"label": yv, "x": xv}).groupby("label"):
                classwise_rows.append({"dataset": dataset_name, "feature": feature, "label": label, "mean_or_rate": float(sub["x"].mean()), "count": int(len(sub))})
        try:
            mi = float(mutual_info_classif(xv.values.reshape(-1, 1), yid, random_state=RANDOM_SEED, discrete_features=is_binary)[0])
        except Exception:
            mi = np.nan
        spearman = np.nan
        if SCIPY_AVAILABLE and xv.nunique() > 1:
            spearman = float(st.spearmanr(xv, yid, nan_policy="omit").correlation)
        means = pd.DataFrame({"label": yv, "x": xv}).groupby("label")["x"].mean().sort_values()
        if len(means) >= 2:
            best_pair = f"{means.index[0]} vs {means.index[-1]}"
        else:
            best_pair = ""
        assoc_rows.append({
            "dataset": dataset_name,
            "feature": feature,
            "phenomenon_group": FEATURE_GROUPS.get(feature, "Unknown"),
            "feature_type": "binary" if is_binary else "continuous",
            "test": test_name,
            "p_value": p_value,
            "effect_size": effect_size,
            "mutual_information": mi,
            "spearman_label_id": spearman,
            "best_separating_emotion_pair": best_pair,
        })
        plt.figure(figsize=(5, 3))
        plot_df = pd.DataFrame({"label": yv, feature: xv})
        if is_binary:
            plot_df.groupby("label")[feature].mean().reindex([l for l in LABEL_ORDER if l in set(yv)]).plot(kind="bar")
            plt.ylabel("event rate")
        else:
            if SEABORN_AVAILABLE:
                sns.boxplot(data=plot_df, x="label", y=feature, order=[l for l in LABEL_ORDER if l in set(yv)])
            else:
                plot_df.boxplot(column=feature, by="label")
            plt.ylabel(feature)
        plt.title(f"{dataset_name}: {feature} by emotion")
        plt.tight_layout()
        plt.savefig(ASSOC_PLOT_DIR / f"{dataset_name}_{feature}.png", dpi=120)
        plt.close()

emotion_association = pd.DataFrame(assoc_rows)
emotion_classwise_stats = pd.DataFrame(classwise_rows)
emotion_association.to_csv(OUT_DIR / "emotion_association.csv", index=False)
emotion_classwise_stats.to_csv(OUT_DIR / "emotion_classwise_stats.csv", index=False)
emotion_association.sort_values(["dataset", "mutual_information"], ascending=[True, False]).head(20)


,dataset,feature,phenomenon_group,feature_type,test,p_value,effect_size,mutual_information,spearman_label_id,best_separating_emotion_pair
32,IEMOCAP,silence_density_10s,Dialogue Rhythm,continuous,kruskal_wallis_epsilon2,9.391241e-37,0.022274,0.152303,0.073958,neutral vs happy
10,IEMOCAP,previous_mean_gap,Response Dynamics,continuous,kruskal_wallis_epsilon2,1.142440e-98,0.061529,0.089762,0.014690,neutral vs sad
26,IEMOCAP,overlap_frequency_window5,Overlap / Interruption,continuous,kruskal_wallis_epsilon2,2.482302e-145,0.088903,0.064200,-0.251216,sad vs angry
28,IEMOCAP,interruption_frequency_window5,Overlap / Interruption,continuous,kruskal_wallis_epsilon2,2.482302e-145,0.088903,0.064200,-0.251216,sad vs angry
1,IEMOCAP,gap_prev,Basic,continuous,kruskal_wallis_epsilon2,8.525995e-66,0.040124,0.063567,0.176945,angry vs sad
41,IEMOCAP,speaker_prev_mean_gap,Speaker Behavior,continuous,kruskal_wallis_epsilon2,6.211415e-84,0.051256,0.060937,0.060045,neutral vs sad
12,IEMOCAP,window5_average_gap,Response Dynamics,continuous,kruskal_wallis_epsilon2,2.741774e-72,0.044109,0.059668,0.146060,angry vs sad
11,IEMOCAP,window3_average_gap,Response Dynamics,continuous,kruskal_wallis_epsilon2,4.023580e-73,0.044621,0.056402,0.162479,angry vs sad
25,IEMOCAP,overlap_frequency_window3,Overlap / Interruption,continuous,kruskal_wallis_epsilon2,2.342025e-120,0.073594,0.045903,-0.229542,sad vs angry
27,IEMOCAP,interruption_frequency_window3,Overlap / Interruption,continuous,kruskal_wallis_epsilon2,2.342025e-120,0.073594,0.045903,-0.229542,sad vs angry


## 7. Redundancy Analysis


In [11]:
redundant_cluster_rows = []
for dataset_name, frame in features_all.groupby("dataset", sort=False):
    numeric = frame[feature_cols].apply(pd.to_numeric, errors="coerce")
    numeric = numeric.loc[:, numeric.notna().any(axis=0)]
    if numeric.empty:
        continue
    spearman = numeric.corr(method="spearman")
    pearson = numeric.corr(method="pearson")
    spearman.to_csv(OUT_DIR / f"redundancy_matrix_spearman_{dataset_name}.csv")
    pearson.to_csv(OUT_DIR / f"redundancy_matrix_pearson_{dataset_name}.csv")
    for i, f1 in enumerate(spearman.columns):
        for f2 in spearman.columns[i+1:]:
            corr = spearman.loc[f1, f2]
            if pd.notna(corr) and abs(corr) > 0.90:
                redundant_cluster_rows.append({"dataset": dataset_name, "feature_a": f1, "feature_b": f2, "spearman_abs": abs(float(corr)), "phenomenon_a": FEATURE_GROUPS.get(f1), "phenomenon_b": FEATURE_GROUPS.get(f2)})
    plt.figure(figsize=(12, 10))
    if SEABORN_AVAILABLE:
        sns.heatmap(spearman, cmap="vlag", center=0, vmin=-1, vmax=1)
    else:
        plt.imshow(spearman, vmin=-1, vmax=1, cmap="coolwarm")
        plt.colorbar()
        plt.xticks(range(len(spearman.columns)), spearman.columns, rotation=90, fontsize=6)
        plt.yticks(range(len(spearman.index)), spearman.index, fontsize=6)
    plt.title(f"Spearman redundancy: {dataset_name}")
    plt.tight_layout()
    plt.savefig(REDUNDANCY_PLOT_DIR / f"spearman_{dataset_name}.png", dpi=140)
    plt.close()

redundant_clusters = pd.DataFrame(redundant_cluster_rows)
for dataset_name in sorted(features_all["dataset"].dropna().unique()):
    redundant_clusters[redundant_clusters["dataset"] == dataset_name].to_csv(OUT_DIR / f"redundant_feature_clusters_{dataset_name}.csv", index=False)
redundant_clusters.head(20)


,dataset,feature_a,feature_b,spearman_abs,phenomenon_a,phenomenon_b
0,IEMOCAP,gap_prev,overlap_duration,0.952256,Basic,Overlap / Interruption
1,IEMOCAP,gap_prev,overlap_ratio,0.912505,Basic,Overlap / Interruption
2,IEMOCAP,overlap_duration,overlap_ratio,0.958256,Overlap / Interruption,Overlap / Interruption
3,IEMOCAP,overlap_duration,overlap_flag,0.906967,Overlap / Interruption,Overlap / Interruption
4,IEMOCAP,overlap_duration,interruption_flag,0.906967,Overlap / Interruption,Overlap / Interruption
5,IEMOCAP,overlap_ratio,overlap_flag,0.906761,Overlap / Interruption,Overlap / Interruption
6,IEMOCAP,overlap_ratio,interruption_flag,0.906761,Overlap / Interruption,Overlap / Interruption
7,IEMOCAP,log_turn_index,speaker_prev_turn_count,0.969887,Basic,Speaker Behavior
8,IEMOCAP,window5_gap_variance,rhythm_variance_window5,1.000000,Response Dynamics,Dialogue Rhythm
9,IEMOCAP,speaker_switch,same_speaker_continuation,0.953926,Turn-taking,Turn-taking


## 8. Complementarity with WavLM Embeddings


In [12]:
def load_wavlm_embeddings():
    if not TORCH_AVAILABLE:
        return pd.DataFrame(), None
    candidates = [PROJECT_ROOT / "results" / "wavlm_shared" / "cache" / "wavlm_mean_embeddings.pt"] + sorted((PROJECT_ROOT / "results").glob("**/cache/*.pt"))
    for path in candidates:
        if not path.exists():
            continue
        try:
            obj = torch.load(path, map_location="cpu")
        except Exception:
            continue
        rows = obj.get("rows_by_utterance") if isinstance(obj, dict) else None
        if not isinstance(rows, dict):
            continue
        records = []
        embeddings = []
        for utt, row in rows.items():
            emb = row.get("embedding") if isinstance(row, dict) else None
            if emb is None:
                continue
            embeddings.append(emb.detach().cpu().numpy() if hasattr(emb, "detach") else np.asarray(emb))
            records.append({"utterance_id": row.get("utterance_id", utt), "dialogue_id": row.get("dialogue_id", ""), "label": row.get("label_name", row.get("label", ""))})
        if embeddings:
            return pd.DataFrame(records), np.vstack(embeddings)
    return pd.DataFrame(), None


def metric_dict(y_true, y_pred):
    return {
        "WA": accuracy_score(y_true, y_pred),
        "UA": balanced_accuracy_score(y_true, y_pred),
        "Macro-F1": f1_score(y_true, y_pred, average="macro"),
        "WF1": f1_score(y_true, y_pred, average="weighted"),
    }

emb_meta, emb_matrix = load_wavlm_embeddings()
probe_rows, importance_rows = [], []
if emb_matrix is None or emb_meta.empty:
    warnings.warn("No WavLM embedding cache found; complementarity probe skipped.")
else:
    feat_subset = features_all.merge(emb_meta[["utterance_id"]].assign(_emb_idx=np.arange(len(emb_meta))), on="utterance_id", how="inner")
    if feat_subset.empty:
        warnings.warn("Embedding cache found but no utterance_id overlap with features_all.")
    else:
        temporal_features = [f for f in feature_cols if features_all[f].notna().mean() > 0.7]
        for dataset_name, frame in feat_subset.groupby("dataset", sort=False):
            if frame["label"].nunique() < 2 or len(frame) < 50:
                continue
            y = frame["label"].astype(str).values
            idx = frame["_emb_idx"].astype(int).values
            X_wav = emb_matrix[idx]
            X_tmp = frame[temporal_features].apply(pd.to_numeric, errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(0).values
            stratify = y if min(pd.Series(y).value_counts()) >= 2 else None
            train_idx, test_idx = train_test_split(np.arange(len(y)), test_size=0.25, random_state=RANDOM_SEED, stratify=stratify)
            probes = {
                "WavLM only": X_wav,
                "Temporal only": X_tmp,
                "WavLM + Temporal": np.hstack([X_wav, X_tmp]),
            }
            fitted_temporal = None
            for probe_name, X in probes.items():
                clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, class_weight="balanced", random_state=RANDOM_SEED))
                clf.fit(X[train_idx], y[train_idx])
                pred = clf.predict(X[test_idx])
                row = {"dataset": dataset_name, "probe": probe_name, **metric_dict(y[test_idx], pred), "n": len(y)}
                probe_rows.append(row)
                if probe_name == "Temporal only":
                    fitted_temporal = clf
            if fitted_temporal is not None and len(temporal_features) > 0:
                try:
                    perm = permutation_importance(fitted_temporal, X_tmp[test_idx], y[test_idx], n_repeats=5, random_state=RANDOM_SEED, scoring="f1_macro")
                    for feature, mean, std in zip(temporal_features, perm.importances_mean, perm.importances_std):
                        importance_rows.append({"dataset": dataset_name, "feature": feature, "permutation_importance_mean": float(mean), "permutation_importance_std": float(std)})
                except Exception as exc:
                    warnings.warn(f"Permutation importance failed for {dataset_name}: {exc}")

complementarity_probe_metrics = pd.DataFrame(probe_rows)
if not complementarity_probe_metrics.empty:
    base = complementarity_probe_metrics.pivot(index="dataset", columns="probe", values=["Macro-F1", "UA"])
    for i, row in complementarity_probe_metrics.iterrows():
        if row["probe"] == "WavLM + Temporal" and (row["dataset"], "WavLM only"):
            wav = complementarity_probe_metrics[(complementarity_probe_metrics["dataset"] == row["dataset"]) & (complementarity_probe_metrics["probe"] == "WavLM only")]
            if not wav.empty:
                complementarity_probe_metrics.loc[i, "delta_macro_f1_vs_wavlm"] = row["Macro-F1"] - float(wav.iloc[0]["Macro-F1"])
                complementarity_probe_metrics.loc[i, "delta_UA_vs_wavlm"] = row["UA"] - float(wav.iloc[0]["UA"])

temporal_probe_feature_importance = pd.DataFrame(importance_rows)
complementarity_probe_metrics.to_csv(OUT_DIR / "complementarity_probe_metrics.csv", index=False)
temporal_probe_feature_importance.to_csv(OUT_DIR / "temporal_probe_feature_importance.csv", index=False)
complementarity_probe_metrics


,dataset,probe,WA,UA,Macro-F1,WF1,n,delta_macro_f1_vs_wavlm,delta_UA_vs_wavlm
0,IEMOCAP,WavLM only,0.635575,0.648769,0.641145,0.634021,5531,NaN,NaN
1,IEMOCAP,Temporal only,0.469993,0.479705,0.467704,0.468234,5531,NaN,NaN
2,IEMOCAP,WavLM + Temporal,0.658713,0.667020,0.664966,0.658643,5531,0.023821,0.018251


## 9. Relationship with Model Errors


In [13]:
def load_named_predictions():
    candidates = {
        "mal": [PROJECT_ROOT / "results" / "wavlm_mal_no_tim" / "predictions.csv", PROJECT_ROOT / "results" / "fair_3phase" / "wavlm_mal" / "predictions.csv"],
        "tim": [PROJECT_ROOT / "results" / "wavlm_tim" / "predictions.csv"],
        "dual": [PROJECT_ROOT / "results" / "dual_branch" / "predictions.csv"],
    }
    out = {}
    for name, paths in candidates.items():
        for path in paths:
            if path.exists():
                frame = pd.read_csv(path)
                if {"dialogue_id", "utterance_id"}.issubset(frame.columns):
                    pred_col = "pred_label" if "pred_label" in frame.columns else "prediction" if "prediction" in frame.columns else None
                    true_col = "true_label" if "true_label" in frame.columns else "gold_label" if "gold_label" in frame.columns else "label" if "label" in frame.columns else None
                    if pred_col and true_col:
                        out[name] = frame[["dialogue_id", "utterance_id", true_col, pred_col]].rename(columns={true_col: "true_label", pred_col: f"{name}_pred"})
                        break
    return out

preds = load_named_predictions()
error_feature_rows, error_classwise_rows = [], []
if {"mal", "dual"}.issubset(preds):
    merged = features_all.merge(preds["mal"], on=["dialogue_id", "utterance_id"], how="inner")
    merged = merged.merge(preds.get("tim", pd.DataFrame(columns=["dialogue_id", "utterance_id", "tim_pred"])), on=["dialogue_id", "utterance_id"], how="left")
    merged = merged.merge(preds["dual"], on=["dialogue_id", "utterance_id"], how="inner", suffixes=("", "_dual"))
    merged["true_label"] = merged["true_label"].astype(str)
    merged["mal_correct"] = merged["mal_pred"].astype(str) == merged["true_label"]
    if "tim_pred" in merged.columns:
        merged["tim_correct"] = merged["tim_pred"].astype(str) == merged["true_label"]
    merged["dual_correct"] = merged["dual_pred"].astype(str) == merged["true_label"]
    merged["dual_improves_over_mal"] = (~merged["mal_correct"]) & merged["dual_correct"]
    merged["dual_hurts_vs_mal"] = merged["mal_correct"] & (~merged["dual_correct"])
    merged["error_group"] = np.select(
        [merged["dual_improves_over_mal"], merged["dual_hurts_vs_mal"], merged["mal_correct"] & merged["dual_correct"]],
        ["MAL wrong / Dual correct", "MAL correct / Dual wrong", "both correct"],
        default="both wrong",
    )
    for feature in feature_cols:
        x = pd.to_numeric(merged[feature], errors="coerce")
        valid = x.notna()
        if valid.sum() < 20:
            continue
        for group_name, sub in merged[valid].groupby("error_group"):
            error_classwise_rows.append({"feature": feature, "error_group": group_name, "mean_or_rate": float(pd.to_numeric(sub[feature], errors="coerce").mean()), "count": int(len(sub))})
        y_imp = merged.loc[valid, "dual_improves_over_mal"].astype(int)
        y_hurt = merged.loc[valid, "dual_hurts_vs_mal"].astype(int)
        xv = x[valid]
        try:
            mi_imp = float(mutual_info_classif(xv.values.reshape(-1, 1), y_imp, random_state=RANDOM_SEED, discrete_features=is_binary_series(xv))[0])
            mi_hurt = float(mutual_info_classif(xv.values.reshape(-1, 1), y_hurt, random_state=RANDOM_SEED, discrete_features=is_binary_series(xv))[0])
        except Exception:
            mi_imp = mi_hurt = np.nan
        imp_mean = float(xv[y_imp == 1].mean()) if y_imp.sum() else np.nan
        hurt_mean = float(xv[y_hurt == 1].mean()) if y_hurt.sum() else np.nan
        other_mean = float(xv[(y_imp == 0) & (y_hurt == 0)].mean()) if ((y_imp == 0) & (y_hurt == 0)).sum() else np.nan
        error_feature_rows.append({
            "feature": feature,
            "phenomenon_group": FEATURE_GROUPS.get(feature),
            "mi_with_dual_improves_over_mal": mi_imp,
            "mi_with_dual_hurts_vs_mal": mi_hurt,
            "improve_mean_or_rate": imp_mean,
            "hurt_mean_or_rate": hurt_mean,
            "other_mean_or_rate": other_mean,
            "improve_minus_hurt": imp_mean - hurt_mean if pd.notna(imp_mean) and pd.notna(hurt_mean) else np.nan,
        })
else:
    warnings.warn(f"Could not load both MAL and Dual prediction files. Found: {sorted(preds)}")

error_feature_analysis = pd.DataFrame(error_feature_rows)
error_feature_classwise = pd.DataFrame(error_classwise_rows)
error_feature_analysis.to_csv(OUT_DIR / "error_feature_analysis.csv", index=False)
error_feature_classwise.to_csv(OUT_DIR / "error_feature_classwise.csv", index=False)
error_feature_analysis.sort_values("mi_with_dual_improves_over_mal", ascending=False).head(20) if not error_feature_analysis.empty else error_feature_analysis


,feature,phenomenon_group,mi_with_dual_improves_over_mal,mi_with_dual_hurts_vs_mal,improve_mean_or_rate,hurt_mean_or_rate,other_mean_or_rate,improve_minus_hurt
18,turn_holding_duration_so_far,Turn-taking,0.011547,0.006665,53.558701,51.621384,66.619927,1.937317
36,speaker_prev_total_speaking_time,Speaker Behavior,0.011547,0.006665,53.558701,51.621384,66.619927,1.937317
40,speaker_prev_interruption_rate,Speaker Behavior,0.008576,0.011589,0.506355,0.498961,0.551267,0.007394
39,speaker_prev_overlap_rate,Speaker Behavior,0.008576,0.011589,0.506355,0.498961,0.551267,0.007394
38,speaker_dominance_turns_so_far,Speaker Behavior,0.008322,0.011613,0.484185,0.525176,0.486158,-0.040991
35,speaker_prev_turn_count,Speaker Behavior,0.006798,0.000666,11.623762,10.827068,14.997175,0.796695
43,speaker_persistence_so_far,Speaker Behavior,0.003726,0.000000,0.237726,0.249040,0.227667,-0.011315
13,window3_gap_variance,Response Dynamics,0.003505,0.005097,16.692380,9.817768,8.171146,6.874612
24,consecutive_overlap_count,Overlap / Interruption,0.002374,0.000000,1.851485,1.616541,1.923023,0.234944
37,speaker_dominance_time_so_far,Speaker Behavior,0.002169,0.012895,0.472011,0.528215,0.498106,-0.056204


## 10. Feature Qualification Table


In [14]:
def max_abs_corr_for_feature(feature: str) -> float:
    vals = []
    for path in OUT_DIR.glob("redundancy_matrix_spearman_*.csv"):
        mat = pd.read_csv(path, index_col=0)
        if feature in mat.columns:
            s = mat[feature].drop(labels=[feature], errors="ignore").abs().dropna()
            if not s.empty:
                vals.append(float(s.max()))
    return max(vals) if vals else np.nan

assoc_mean = emotion_association.groupby("feature").agg(effect_size=("effect_size", "max"), mutual_information=("mutual_information", "max")).reset_index() if not emotion_association.empty else pd.DataFrame(columns=["feature", "effect_size", "mutual_information"])
dist_mean = distribution_stats.groupby("feature").agg(missing_rate=("missing_rate", "mean"), near_constant=("near_constant", "max"), high_missing=("high_missing", "max"), sparse_event=("sparse_event", "max"), event_rate_or_variance=("binary_rate", "mean"), std=("std", "mean")).reset_index() if not distribution_stats.empty else pd.DataFrame()
err_mean = error_feature_analysis.groupby("feature").agg(error_explanation_score=("mi_with_dual_improves_over_mal", "max"), hurt_score=("mi_with_dual_hurts_vs_mal", "max")).reset_index() if not error_feature_analysis.empty else pd.DataFrame(columns=["feature", "error_explanation_score", "hurt_score"])
imp_mean = temporal_probe_feature_importance.groupby("feature").agg(complementarity_score=("permutation_importance_mean", "max")).reset_index() if not temporal_probe_feature_importance.empty else pd.DataFrame(columns=["feature", "complementarity_score"])

qualification_rows = []
for feature, group in FEATURE_GROUPS.items():
    dist = dist_mean[dist_mean["feature"] == feature]
    assoc = assoc_mean[assoc_mean["feature"] == feature]
    err = err_mean[err_mean["feature"] == feature]
    imp = imp_mean[imp_mean["feature"] == feature]
    missing_rate = float(dist["missing_rate"].iloc[0]) if not dist.empty else np.nan
    near_constant = bool(dist["near_constant"].iloc[0]) if not dist.empty else False
    high_missing = bool(dist["high_missing"].iloc[0]) if not dist.empty else True
    sparse_event = bool(dist["sparse_event"].iloc[0]) if not dist.empty else False
    event_or_var = float(dist["event_rate_or_variance"].iloc[0]) if not dist.empty and pd.notna(dist["event_rate_or_variance"].iloc[0]) else (float(dist["std"].iloc[0]) if not dist.empty else np.nan)
    effect = float(assoc["effect_size"].iloc[0]) if not assoc.empty and pd.notna(assoc["effect_size"].iloc[0]) else 0.0
    mi = float(assoc["mutual_information"].iloc[0]) if not assoc.empty and pd.notna(assoc["mutual_information"].iloc[0]) else 0.0
    redundancy = max_abs_corr_for_feature(feature)
    complementarity = float(imp["complementarity_score"].iloc[0]) if not imp.empty and pd.notna(imp["complementarity_score"].iloc[0]) else 0.0
    error_score = float(err["error_explanation_score"].iloc[0]) if not err.empty and pd.notna(err["error_explanation_score"].iloc[0]) else 0.0
    hurt_score = float(err["hurt_score"].iloc[0]) if not err.empty and pd.notna(err["hurt_score"].iloc[0]) else 0.0
    i_comp = feasibility[feasibility["feature_name"].str.contains(feature.replace("_duration", ""), regex=False, na=False)]["computable_on_IEMOCAP"].iloc[0] if not feasibility[feasibility["feature_name"].str.contains(feature.replace("_duration", ""), regex=False, na=False)].empty else "yes"
    m_comp = feasibility[feasibility["feature_name"].str.contains(feature.replace("_duration", ""), regex=False, na=False)]["computable_on_MELD"].iloc[0] if not feasibility[feasibility["feature_name"].str.contains(feature.replace("_duration", ""), regex=False, na=False)].empty else "partial"
    if high_missing:
        dist_quality = "low"
    elif sparse_event:
        dist_quality = "medium"
    elif near_constant:
        dist_quality = "low"
    else:
        dist_quality = "high"
    evidence = mi + effect + max(0.0, complementarity) + error_score
    if i_comp == "no" and m_comp == "no":
        rec, reason = "Not computable", "required metadata unavailable"
    elif high_missing or near_constant:
        rec, reason = "Noisy", "high missing rate or near-constant distribution"
    elif pd.notna(redundancy) and redundancy > 0.90 and evidence < 0.03:
        rec, reason = "Redundant", f"high redundancy (max |rho|={redundancy:.2f}) with weak independent evidence"
    elif hurt_score > error_score and hurt_score > 0.02:
        rec, reason = "Noisy", "more associated with Dual hurting than improving over MAL"
    elif evidence >= 0.05 and not (pd.notna(redundancy) and redundancy > 0.95):
        rec, reason = "Highly Recommended", "nontrivial association/complementarity/error evidence with usable distribution"
    elif evidence >= 0.015 or sparse_event:
        rec, reason = "Useful", "moderate evidence or sparse but meaningful event feature"
    else:
        rec, reason = "Weak", "computable but weak current evidence"
    qualification_rows.append({
        "feature_name": feature,
        "phenomenon_group": group,
        "description": feature.replace("_", " "),
        "literature_motivation_short": "captures conversational timing, turn-taking, overlap, speaker behavior, or dialogue rhythm relevant to emotion dynamics",
        "computable_iemocap": i_comp,
        "computable_meld": m_comp,
        "causal": True,
        "online_suitable": True,
        "distribution_quality": dist_quality,
        "missing_rate": missing_rate,
        "event_rate_or_variance": event_or_var,
        "emotion_association_score": mi + effect,
        "effect_size": effect,
        "mutual_information": mi,
        "redundancy_score": redundancy,
        "complementarity_score": complementarity,
        "error_explanation_score": error_score,
        "final_recommendation": rec,
        "recommendation_reason": reason,
    })

feature_qualification_table = pd.DataFrame(qualification_rows)
recommended_features = feature_qualification_table[feature_qualification_table["final_recommendation"].isin(["Highly Recommended", "Useful"])]
feature_qualification_table.to_csv(OUT_DIR / "feature_qualification_table.csv", index=False)
recommended_features.to_csv(OUT_DIR / "recommended_features.csv", index=False)
feature_qualification_table.sort_values(["final_recommendation", "emotion_association_score"], ascending=[True, False]).head(30)


,feature_name,phenomenon_group,description,literature_motivation_short,computable_iemocap,computable_meld,causal,online_suitable,distribution_quality,missing_rate,event_rate_or_variance,emotion_association_score,effect_size,mutual_information,redundancy_score,complementarity_score,error_explanation_score,final_recommendation,recommendation_reason
45,conflict_like_state,Dialogue State,conflict like state,"captures conversational timing, turn-taking, o...",yes,partial,True,True,high,0.00000,0.413404,0.274515,0.245756,0.028758,0.737709,0.002204,1.180875e-04,Highly Recommended,nontrivial association/complementarity/error e...
44,rapid_exchange_state,Dialogue State,rapid exchange state,"captures conversational timing, turn-taking, o...",yes,partial,True,True,high,0.00000,0.371628,0.220417,0.199970,0.020446,0.806695,0.009291,1.290787e-04,Highly Recommended,nontrivial association/complementarity/error e...
32,silence_density_10s,Dialogue Rhythm,silence density 10s,"captures conversational timing, turn-taking, o...",yes,partial,True,True,high,0.00000,2.137058,0.174577,0.022274,0.152303,0.870923,0.066665,0.000000e+00,Highly Recommended,nontrivial association/complementarity/error e...
10,previous_mean_gap,Response Dynamics,previous mean gap,"captures conversational timing, turn-taking, o...",yes,partial,True,True,high,0.06226,4.550859,0.151291,0.061529,0.089762,0.812375,0.057435,9.785346e-04,Highly Recommended,nontrivial association/complementarity/error e...
46,hesitation_state,Dialogue State,hesitation state,"captures conversational timing, turn-taking, o...",yes,partial,True,True,high,0.00000,0.379770,0.126029,0.119011,0.007019,0.762831,0.000089,3.982749e-07,Highly Recommended,nontrivial association/complementarity/error e...
41,speaker_prev_mean_gap,Speaker Behavior,speaker prev mean gap,"captures conversational timing, turn-taking, o...",yes,partial,True,True,high,0.00000,6.519493,0.112192,0.051256,0.060937,0.812375,0.017057,2.090248e-03,Highly Recommended,nontrivial association/complementarity/error e...
12,window5_average_gap,Response Dynamics,window5 average gap,"captures conversational timing, turn-taking, o...",yes,partial,True,True,high,0.00000,5.729670,0.103778,0.044109,0.059668,0.870923,0.057802,0.000000e+00,Highly Recommended,nontrivial association/complementarity/error e...
11,window3_average_gap,Response Dynamics,window3 average gap,"captures conversational timing, turn-taking, o...",yes,partial,True,True,high,0.00000,6.807862,0.101023,0.044621,0.056402,0.863023,0.004710,0.000000e+00,Highly Recommended,nontrivial association/complementarity/error e...
8,long_pause,Response Dynamics,long pause,"captures conversational timing, turn-taking, o...",yes,partial,True,True,high,0.00000,0.240462,0.100190,0.095718,0.004472,0.766861,0.007201,3.624474e-05,Highly Recommended,nontrivial association/complementarity/error e...
6,immediate_response,Response Dynamics,immediate response,"captures conversational timing, turn-taking, o...",yes,partial,True,True,high,0.00000,0.117985,0.072204,0.069880,0.002324,0.798235,0.007877,9.073403e-05,Highly Recommended,nontrivial association/complementarity/error e...


## 11. Final Recommendation for TIM v2


In [16]:
def top_table_markdown(frame, cols, n=12):
    if frame is None or frame.empty:
        return "No data available."
    return frame[cols].head(n).to_markdown(index=False)

feas_summary = feasibility.groupby(["computable_on_IEMOCAP", "computable_on_MELD"]).size().reset_index(name="feature_count")
rec_summary = feature_qualification_table.groupby(["phenomenon_group", "final_recommendation"]).size().reset_index(name="count")
recommended_by_group = recommended_features.groupby("phenomenon_group")["feature_name"].apply(lambda s: ", ".join(s.head(10))).reset_index() if not recommended_features.empty else pd.DataFrame(columns=["phenomenon_group", "feature_name"])

report = f"""
# Feature Engineering Report for TIM v2

## 1. Executive summary

This notebook loaded available IEMOCAP and MELD metadata, constructed causal temporal interaction features, and evaluated feature feasibility, distribution quality, emotion association, redundancy, complementarity with WavLM embeddings, and relation to MAL/DualBranch error patterns.

Loaded rows by dataset:

{df_all.groupby('dataset').size().to_markdown() if not df_all.empty else 'No metadata loaded.'}

## 2. Dataset feasibility comparison: IEMOCAP vs MELD

{feas_summary.to_markdown(index=False) if not feas_summary.empty else 'No feasibility data.'}

IEMOCAP is the stronger dataset for timestamp-based TIM because it provides dialogue-level start/end timing with audio utterances. MELD can provide start/end columns when local CSVs include `StartTime` and `EndTime`, but these timestamps are less central to the dataset and should be treated as partially supported unless verified against the media files.

## 3. Which features are actually measurable

{top_table_markdown(feasibility, ['feature_name', 'phenomenon_group', 'computable_on_IEMOCAP', 'computable_on_MELD', 'reason' if 'reason' in feasibility.columns else 'iemocap_reason'], 20)}

## 4. Which interaction phenomena appear frequently enough

Distribution flags are stored in `feature_distribution_stats.csv`. Near-constant, sparse-event, and high-missing flags should be used before adding features to a model.

Top distribution warnings:

{top_table_markdown(distribution_stats.sort_values(['high_missing', 'near_constant', 'sparse_event'], ascending=False), ['dataset', 'feature', 'missing_rate', 'zero_rate', 'binary_rate', 'near_constant', 'sparse_event', 'high_missing'], 20) if not distribution_stats.empty else 'No distribution stats.'}

## 5. Which features are statistically associated with emotion

{top_table_markdown(emotion_association.sort_values('mutual_information', ascending=False), ['dataset', 'feature', 'phenomenon_group', 'effect_size', 'mutual_information', 'p_value', 'best_separating_emotion_pair'], 20) if not emotion_association.empty else 'No emotion association stats.'}

## 6. Which features are complementary to WavLM

{complementarity_probe_metrics.to_markdown(index=False) if not complementarity_probe_metrics.empty else 'WavLM embedding cache unavailable or no overlap; complementarity probe skipped.'}

Feature importance from the temporal probe is saved to `temporal_probe_feature_importance.csv`.

## 7. Which features explain MAL/Dual errors

{top_table_markdown(error_feature_analysis.sort_values('mi_with_dual_improves_over_mal', ascending=False), ['feature', 'phenomenon_group', 'mi_with_dual_improves_over_mal', 'mi_with_dual_hurts_vs_mal', 'improve_minus_hurt'], 20) if not error_feature_analysis.empty else 'MAL/Dual prediction files unavailable or not mergeable; error-feature analysis skipped.'}

## 8. Recommended feature groups for TIM v2

{recommended_by_group.to_markdown(index=False) if not recommended_by_group.empty else 'No recommended features under current thresholds.'}

## 9. Features to remove or avoid

Avoid features marked `Noisy`, `Not computable`, or `Redundant` in `feature_qualification_table.csv`. In particular, sparse overlap/interruption features should be included only as grouped event signals or through gated branches, not as dominant standalone scalars.

## 10. Proposed TIM v2 design based on the evidence

Recommended TIM v2 design:

- Response dynamics group: latency/gap level, short response, long pause, rolling gap mean/variance.
- Turn-taking group: speaker switch, same-speaker continuation, consecutive turns, local switch frequency.
- Overlap/interruption group: overlap duration/ratio/flag, interruption flag, overlap frequencies, competitive/cooperative overlap proxies.
- Speaker behavior group: speaker dominance by time/turns, historical overlap/interruption tendency, speaker persistence.
- Dialogue rhythm/state group: interaction density, silence density, burstiness, rhythm variance, rapid/conflict/hesitation/calm/floor-competition states.
- Use group-wise encoders so noisy sparse groups do not dominate dense response/rhythm features.
- Use adaptive gates to suppress temporal groups when they are unavailable, sparse, or contradictory to acoustic evidence.
- Keep dual-branch fusion: acoustic dialogue memory and temporal-interaction memory should contribute separate residuals before final classification.
"""

(OUT_DIR / "feature_engineering_report.md").write_text(report.strip() + "\n", encoding="utf-8")
print(report[:3000])



# Feature Engineering Report for TIM v2

## 1. Executive summary

This notebook loaded available IEMOCAP and MELD metadata, constructed causal temporal interaction features, and evaluated feature feasibility, distribution quality, emotion association, redundancy, complementarity with WavLM embeddings, and relation to MAL/DualBranch error patterns.

Loaded rows by dataset:

| dataset   |     0 |
|:----------|------:|
| IEMOCAP   |  7529 |
| MELD      | 13708 |

## 2. Dataset feasibility comparison: IEMOCAP vs MELD

| computable_on_IEMOCAP   | computable_on_MELD   |   feature_count |
|:------------------------|:---------------------|----------------:|
| yes                     | partial              |              31 |
| yes                     | yes                  |               5 |

IEMOCAP is the stronger dataset for timestamp-based TIM because it provides dialogue-level start/end timing with audio utterances. MELD can provide start/end columns when local CSVs include `StartTime` 

## 12. Save All Outputs


In [17]:
outputs = sorted(p.relative_to(OUT_DIR) for p in OUT_DIR.rglob("*") if p.is_file())
print(f"Saved {len(outputs)} files under {OUT_DIR}")
for p in outputs[:80]:
    print(p)


Saved 217 files under /Users/ngocbao/Documents/Document/research/main/speech/exps/demo/results/feature_engineering
complementarity_probe_metrics.csv
emotion_association.csv
emotion_classwise_stats.csv
error_feature_analysis.csv
error_feature_classwise.csv
feature_distribution_stats.csv
feature_engineering_report.md
feature_feasibility.csv
feature_qualification_table.csv
features_all.csv
metadata_all.csv
plots/distributions/IEMOCAP_abs_gap.png
plots/distributions/IEMOCAP_burstiness_window5.png
plots/distributions/IEMOCAP_calm_state.png
plots/distributions/IEMOCAP_competitive_overlap_proxy.png
plots/distributions/IEMOCAP_conflict_like_state.png
plots/distributions/IEMOCAP_consecutive_overlap_count.png
plots/distributions/IEMOCAP_consecutive_same_speaker_turns.png
plots/distributions/IEMOCAP_cooperative_overlap_proxy.png
plots/distributions/IEMOCAP_duration.png
plots/distributions/IEMOCAP_floor_competition_state.png
plots/distributions/IEMOCAP_gap_prev.png
plots/distributions/IEMOCAP_hesi